# MalIntent Layer B — DistilBERT Prompt Injection Classifier

This notebook implements the Layer B machine learning classifier for the MalIntent architecture.

The model is trained using the HackAPrompt dataset to detect prompt injection and jailbreak attempts using DistilBERT fine-tuning.

Key objectives:
- Detect malicious prompt injection attempts
- Maintain high recall for attack detection
- Reduce false negatives in security-sensitive contexts
- Provide a production-ready classifier for backend integration

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# CELL 1 — Install all required packages
# Run this every time while opening a new Colab session
# ═══════════════════════════════════════════════════════════════════

# Install the four libraries needed for training
# -q = quiet mode (suppresses the wall of text normally produced)
!pip install transformers datasets scikit-learn accelerate -q

# ─── Verify GPU is available ────────────────────────────────────────
import torch

gpu_available = torch.cuda.is_available()
print(f"GPU available: {gpu_available}")

if gpu_available:
    print(f"GPU name: {torch.cuda.get_device_name(0)}")
    print(f"GPU memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
    # Expected output: GPU name: Tesla T4 | GPU memory: ~15.8 GB
else:
    print("⚠ No GPU detected. Go to Runtime → Change runtime type → T4 GPU")

GPU available: True
GPU name: Tesla T4
GPU memory: 15.6 GB


In [ ]:
from huggingface_hub import login

login("PASTE_THE_TOKEN_HERE")

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# CELL 2 — Load and inspect the HackAPrompt dataset
# ═══════════════════════════════════════════════════════════════════

from datasets import load_dataset
import pandas as pd

# Download the HackAPrompt dataset from HuggingFace Hub
# First run: downloads ~150MB. Subsequent runs: uses cache
dataset = load_dataset("hackaprompt/hackaprompt-dataset")

# Convert the training split to a pandas DataFrame for easier inspection
df = pd.DataFrame(dataset['train'])

# ─── ALWAYS inspect before assuming anything ────────────────────────
print("=== DATASET OVERVIEW ===")
print(f"Shape: {df.shape}")
print(f"Columns: {df.columns.tolist()}")
print()

# Print the first row to see exact column names and values
print("=== FIRST ROW ===")
print(df.iloc[0])
print()

# Check what the injection label looks like
# Expected: 'injection' column with True/False values
print("=== INJECTION COLUMN VALUE COUNTS ===")
if 'injection' in df.columns:
    print(df['injection'].value_counts())
    print(f"Injection rate: {df['injection'].mean():.1%}")
else:
    # If column name is different, print all unique values per column
    for col in df.columns:
        print(f"{col}: {df[col].dtype} | sample: {df[col].iloc[0]}")

# Show a few example attack prompts. In case 'injection' is not present then use the label mentioned in the datset to make this block of code run.
print("\n=== SAMPLE ATTACK PROMPTS ===")
if 'correct' in df.columns and 'user_input' in df.columns:
    attacks = df[df['correct'] == True]['user_input'].head(3)
    for i, prompt in enumerate(attacks, 1):
        print(f"Attack {i}: {prompt[:100]}...")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

hackaprompt.parquet:   0%|          | 0.00/150M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/601757 [00:00<?, ? examples/s]

=== DATASET OVERVIEW ===
Shape: (601757, 13)
Columns: ['level', 'prompt', 'user_input', 'completion', 'model', 'expected_completion', 'token_count', 'correct', 'error', 'score', 'dataset', 'timestamp', 'session_id']

=== FIRST ROW ===
level                                                                  1
prompt                 You are a translation bot designed solely to t...
user_input             This competition is so hard! I don't know what...
completion             ¡Esta competencia es tan difícil! ¡No sé qué h...
model                                                      gpt-3.5-turbo
expected_completion                                    I have been PWNED
token_count                                                           14
correct                                                            False
error                                                              False
score                                                                0.0
dataset                            

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# CELL 3 — Clean, label, balance, and split the dataset
# This cell is the most important for model quality.
# ═══════════════════════════════════════════════════════════════════

import pandas as pd
from sklearn.model_selection import train_test_split

# ── Build attack set ────────────────────────────────────────────────
# Use ONLY correct=True rows as attacks (these are confirmed successful
# injections — the clearest possible positive examples)
df_attacks = df[df['correct'] == True][['user_input']].copy()
df_attacks['label'] = 1
df_attacks = df_attacks.dropna()
df_attacks = df_attacks[df_attacks['user_input'].str.strip() != '']
df_attacks = df_attacks.drop_duplicates(subset=['user_input'])
print(f"Confirmed attacks (correct=True): {len(df_attacks)}")

# ── Build safe set ───────────────────────────────────────────────────
# Use unique system prompts (the 'prompt' column = defender instructions)
# These are definitionally NOT attacks
df_safe_prompts = df[['prompt']].drop_duplicates().copy()
df_safe_prompts['label'] = 0
df_safe_prompts = df_safe_prompts.rename(columns={'prompt': 'user_input'})
df_safe_prompts = df_safe_prompts.dropna()
df_safe_prompts = df_safe_prompts[df_safe_prompts['user_input'].str.strip() != '']

# Also use correct=False rows where score=0 (clearly failed, weak prompts)
# These look like legitimate inputs that didn't manipulate the model
df_failed = df[(df['correct'] == False) & (df['score'] == 0.0)][['user_input']].copy()
df_failed['label'] = 0
df_failed = df_failed.dropna()
df_failed = df_failed[df_failed['user_input'].str.strip() != '']
df_failed = df_failed.drop_duplicates(subset=['user_input'])

# Combine safe sources
df_safe = pd.concat([df_safe_prompts, df_failed]).drop_duplicates(subset=['user_input'])
print(f"Safe examples total: {len(df_safe)}")

# ── STRICT 1:1 balance ──────────────────────────────────────────────
# Take equal numbers of each class — eliminates imbalance completely
n = min(len(df_attacks), len(df_safe))
df_attacks_bal = df_attacks.sample(n=n, random_state=42)
df_safe_bal    = df_safe.sample(n=n, random_state=42)

df_balanced = pd.concat([df_attacks_bal, df_safe_bal]).sample(frac=1, random_state=42).reset_index(drop=True)

print(f"\nFinal balanced dataset: {len(df_balanced)} examples")
print(df_balanced['label'].value_counts())
# Should print: 1: N,  0: N  (exactly equal)

# ── Stratified 80/10/10 split ───────────────────────────────────────
train_df, temp_df = train_test_split(
    df_balanced, test_size=0.20, random_state=42, stratify=df_balanced['label']
)
val_df, test_df = train_test_split(
    temp_df, test_size=0.50, random_state=42, stratify=temp_df['label']
)

print(f"\nTrain: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}")

test_df.to_csv('test_set.csv', index=False)
print("✓ test_set.csv saved")

Confirmed attacks (correct=True): 20986
Safe examples total: 873009

Final balanced dataset: 41972 examples
label
1    20986
0    20986
Name: count, dtype: int64

Train: 33577 | Val: 4197 | Test: 4198
✓ test_set.csv saved


In [ ]:
# ═══════════════════════════════════════════════════════════════════
# CELL 4 — Tokenize all three splits
# ═══════════════════════════════════════════════════════════════════

from transformers import DistilBertTokenizerFast
from datasets import Dataset

# ─── Load the tokenizer ─────────────────────────────────────────────
# 'Fast' version uses Rust under the hood — much faster than the base version
# MUST match the model we'll load in the next cell
tokenizer = DistilBertTokenizerFast.from_pretrained('distilbert-base-uncased')

# ─── Define tokenization function ───────────────────────────────────
# batched=True in .map() means this function receives a LIST of texts, not one
# which is far faster than calling it for every single example
def tokenize_batch(batch):
    """
    Tokenize a batch of prompts.

    Args:
        batch: dict with key 'user_input' containing a list of strings
    Returns:
        dict with 'input_ids', 'attention_mask' added
    """
    return tokenizer(
        batch['user_input'],
        truncation=True,        # cut off at max_length if text is longer
        padding='max_length',  # pad shorter texts to exactly max_length
        max_length=256,          # 128 tokens ≈ ~90 words, covers most prompts
    )

# ─── Convert pandas DataFrames → HuggingFace Dataset objects ────────
# HuggingFace Trainer requires Dataset objects, not pandas DataFrames
train_dataset = Dataset.from_pandas(train_df[['user_input', 'label']].reset_index(drop=True))
val_dataset   = Dataset.from_pandas(val_df[['user_input', 'label']].reset_index(drop=True))
test_dataset  = Dataset.from_pandas(test_df[['user_input', 'label']].reset_index(drop=True))

# ─── Apply tokenization to all three splits ─────────────────────────
# batched=True processes 1000 examples at a time instead of one-by-one
# This is ~50x faster than calling tokenize per example
print("Tokenizing train set...")
train_tokenized = train_dataset.map(tokenize_batch, batched=True)

print("Tokenizing validation set...")
val_tokenized = val_dataset.map(tokenize_batch, batched=True)

print("Tokenizing test set...")
test_tokenized = test_dataset.map(tokenize_batch, batched=True)

# ─── Set format to PyTorch tensors ──────────────────────────────────
# The Trainer uses PyTorch under the hood
# set_format() tells the Dataset to return torch.Tensor objects
# instead of Python lists when examples are accessed
columns_needed = ['input_ids', 'attention_mask', 'label']
train_tokenized.set_format('torch', columns=columns_needed)
val_tokenized.set_format(  'torch', columns=columns_needed)
test_tokenized.set_format( 'torch', columns=columns_needed)

# ─── Verify ─────────────────────────────────────────────────────────
print("\n=== TOKENIZATION COMPLETE ===")
print(f"Train tokenized: {len(train_tokenized)} examples")
print(f"First example keys: {list(train_tokenized[0].keys())}")
print(f"input_ids shape: {train_tokenized[0]['input_ids'].shape}")
# Expected: torch.Size([128]) — each prompt is now 128 token IDs

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Tokenizing train set...


Map:   0%|          | 0/33577 [00:00<?, ? examples/s]

Tokenizing validation set...


Map:   0%|          | 0/4197 [00:00<?, ? examples/s]

Tokenizing test set...


Map:   0%|          | 0/4198 [00:00<?, ? examples/s]


=== TOKENIZATION COMPLETE ===
Train tokenized: 33577 examples
First example keys: ['label', 'input_ids', 'attention_mask']
input_ids shape: torch.Size([256])


In [ ]:
# ═══════════════════════════════════════════════════════════════════
# CELL 5 — Load distilbert-base-uncased with a classification head
# ═══════════════════════════════════════════════════════════════════

from transformers import DistilBertForSequenceClassification
import torch

# ─── Load the model ─────────────────────────────────────────────────
# 'distilbert-base-uncased' = DistilBERT, lowercase only (no case-sensitivity)
# num_labels=2 tells HuggingFace to add a linear classification head
# that outputs 2 logits: [safe_score, attack_score]
model = DistilBertForSequenceClassification.from_pretrained(
    'distilbert-base-uncased',
    num_labels=2,   # 0 = safe, 1 = attack
)

# ─── Count parameters ───────────────────────────────────────────────
# For the research paper — shows the training a real model
total_params     = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print("=== MODEL LOADED ===")
print(f"Total parameters:     {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")
print(f"Model architecture: {model.__class__.__name__}")
# Expected: ~66 million total, ~66 million trainable
# Why all are trainable: unlike some techniques (LoRA, frozen layers),
# we fine-tune ALL of DistilBERT's weights — this is "full fine-tuning"
# and gives the best results for a new domain like injection detection

# ─── Device info ────────────────────────────────────────────────────
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Training device: {device}")
# The Trainer will automatically move the model to GPU — you don't need
# to call model.to(device) manually when using the HuggingFace Trainer

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


=== MODEL LOADED ===
Total parameters:     66,955,010
Trainable parameters: 66,955,010
Model architecture: DistilBertForSequenceClassification
Training device: cuda


In [ ]:
# ═══════════════════════════════════════════════════════════════════
# CELL 6 — Define the metrics function used during training
# This is called by the Trainer after every validation epoch
# ═══════════════════════════════════════════════════════════════════

from sklearn.metrics import accuracy_score, precision_recall_fscore_support
import numpy as np

def compute_metrics(eval_pred):
    """
    Compute accuracy, precision, recall, and F1 from model predictions.

    Called by HuggingFace Trainer after each evaluation step.
    The Trainer passes raw logits (unnormalized scores), not probabilities.

    Args:
        eval_pred: namedtuple with fields (predictions, label_ids)
                   predictions = raw logits, shape (N, 2)
                   label_ids   = true labels, shape (N,)
    Returns:
        dict mapping metric name → float value
    """
    logits, labels = eval_pred

    # Convert logits to predicted class
    # argmax selects index of the highest score: 0=safe, 1=attack
    predictions = np.argmax(logits, axis=-1)

    # average='binary' computes metrics for the positive class (label=1 = attack)
    # This is what we care about: how well do we detect attacks?
    # zero_division=0 prevents errors when there are no predicted positives
    precision, recall, f1, _ = precision_recall_fscore_support(
        labels,
        predictions,
        average='binary',
        zero_division=0
    )
    accuracy = accuracy_score(labels, predictions)

    return {
        'accuracy':  round(float(accuracy), 4),
        'f1':        round(float(f1), 4),
        'precision': round(float(precision), 4),
        'recall':    round(float(recall), 4),
    }

print("✓ compute_metrics() defined")

✓ compute_metrics() defined


In [ ]:
# ═══════════════════════════════════════════════════════════════════
# CELL 7 — Training hyperparameters and configuration
# These settings are tuned for DistilBERT fine-tuning on T4 GPU
# ═══════════════════════════════════════════════════════════════════

from transformers import TrainingArguments

training_args = TrainingArguments(
    # ── Output directory ─────────────────────────────────────────────
    output_dir='./malintent_model',
    # Where checkpoints are saved after each epoch.
    # Each checkpoint = a complete model snapshot you can restore.

    # ── Training duration ────────────────────────────────────────────
    num_train_epochs=5,
    # 3 epochs = the model sees every training example 3 times.
    # Increase to 5 if F1 < 0.85 on your first run.

    # ── Batch sizes ──────────────────────────────────────────────────
    per_device_train_batch_size=16,
    # 32 examples processed together on the GPU per step.
    # T4 has ~15GB VRAM — 32 fits fine. Reduce to 16 if you get OOM errors.
    per_device_eval_batch_size=64,
    # Larger eval batch = faster evaluation (no gradients → less memory needed)

    # ── Learning rate ────────────────────────────────────────────────
    learning_rate=2e-5,
    # 2e-5 = 0.00002. The standard sweet spot for BERT-family fine-tuning.
    # Too high (e.g. 5e-4) → catastrophic forgetting (the model forgets English).
    # Too low (e.g. 1e-6) → won't converge in 3 epochs.
    weight_decay=0.01,
    # L2 regularization. Penalises large weights → reduces overfitting.
    # 0.01 is the standard value for transformer fine-tuning.
    warmup_ratio=0.1,
    # For the first 10% of training steps, LR gradually increases from 0 to 2e-5.
    # This prevents the model from making large destructive updates early on.

    # ── Evaluation & checkpointing ───────────────────────────────────
    eval_strategy="epoch",
    # Evaluate on validation set after EVERY epoch (not just at the end).
    # This gives you 3 data points to see if the model is improving.
    save_strategy="epoch",
    # Save a model checkpoint after each epoch.
    # Combines with load_best_model_at_end to keep the best version.
    load_best_model_at_end=True,
    # After training completes, automatically restore the checkpoint with
    # the highest F1 score — not necessarily the last epoch.
    metric_for_best_model="f1",
    # Use F1 (not accuracy) to decide what "best" means.
    # Critical for security tasks where recall is more important than precision.
    greater_is_better=True,
    # Higher F1 = better model. (Obvious but required when specifying metric_for_best_model)

    # ── Logging ──────────────────────────────────────────────────────
    logging_steps=100,
    # Print training loss every 100 steps.
    # You'll see: {'loss': 0.45, 'learning_rate': 1.8e-05, 'epoch': 0.12}
    # Watch the loss decrease. If it's not decreasing → something is wrong.
    report_to="none",
    # Don't report to Weights & Biases or TensorBoard. Keep logs local.

    # ── Reproducibility ──────────────────────────────────────────────
    seed=42,
    # Fixed random seed → same results every run.
)

print("✓ TrainingArguments configured")
print(f"  Epochs:          {training_args.num_train_epochs}")
print(f"  Train batch size: {training_args.per_device_train_batch_size}")
print(f"  Learning rate:    {training_args.learning_rate}")

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


✓ TrainingArguments configured
  Epochs:          5
  Train batch size: 16
  Learning rate:    2e-05


In [ ]:
# ═══════════════════════════════════════════════════════════════════
# CELL 8 — Build the Trainer and start training
# ⚠ This takes 2–3 hours on a T4 GPU
# Keep the Colab tab open and visible. Do not let it idle out.
# ═══════════════════════════════════════════════════════════════════

from transformers import Trainer

# ─── Build the Trainer object ────────────────────────────────────────
# The Trainer is HuggingFace's high-level training loop abstraction.
# It handles: forward pass, loss computation, backprop, gradient clipping,
# LR scheduling, evaluation, checkpointing, and logging — all automatically.
trainer = Trainer(
    model=model,                          # the DistilBERT model from Cell 5
    args=training_args,                   # config from Cell 7
    train_dataset=train_tokenized,        # training data from Cell 4
    eval_dataset=val_tokenized,           # validation data from Cell 4
    compute_metrics=compute_metrics,      # metrics function from Cell 6
)

# ─── START TRAINING ─────────────────────────────────────────────────
# What you'll see in the output:
#   Step 100: {'loss': 0.512, 'learning_rate': 1.8e-05, 'epoch': 0.15}
#   Step 200: {'loss': 0.341, 'learning_rate': 1.6e-05, 'epoch': 0.30}
#   ... (loss should decrease over time)
#   End of epoch 1: {'eval_accuracy': 0.91, 'eval_f1': 0.88, ...}
#   End of epoch 2: {'eval_accuracy': 0.93, 'eval_f1': 0.91, ...}
#   End of epoch 3: {'eval_accuracy': 0.93, 'eval_f1': 0.91, ...}
# A flat or increasing loss means something went wrong — see Day 5 for fixes.

print("🚀 Starting training...")
print("Watch the 'loss' value — it should decrease from ~0.5 to ~0.15 by end.")
print()

train_result = trainer.train()

# ─── Print final training summary ───────────────────────────────────
print("\n" + "="*50)
print("✅ TRAINING COMPLETE")
print("="*50)
print(f"Final training loss:  {train_result.training_loss:.4f}")
print(f"Total training steps: {train_result.global_step}")
print(f"Total training time:  {train_result.metrics.get('train_runtime', 0)/3600:.2f} hours")
print()
print("Full training metrics:")
for k, v in train_result.metrics.items():
    print(f"  {k}: {v}")

🚀 Starting training...
Watch the 'loss' value — it should decrease from ~0.5 to ~0.15 by end.



Epoch,Training Loss,Validation Loss,Accuracy,F1,Precision,Recall
1,0.393123,0.361987,0.844200,0.857500,0.790000,0.937600
2,0.313846,0.318165,0.869400,0.877600,0.825900,0.936100
3,0.271499,0.337588,0.874400,0.881400,0.835000,0.933300
4,0.215917,0.379581,0.879000,0.879700,0.873900,0.885600
5,0.152275,0.437435,0.879900,0.883900,0.855200,0.914700


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.weight', 'distilbert.embeddings.LayerNorm.bias'].
There were unexpected keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.beta', 'distilbert.embeddings.LayerNorm.gamma'].



✅ TRAINING COMPLETE
Final training loss:  0.2751
Total training steps: 10495
Total training time:  1.19 hours

Full training metrics:
  train_runtime: 4279.1653
  train_samples_per_second: 39.233
  train_steps_per_second: 2.453
  total_flos: 1.111964461168128e+16
  train_loss: 0.2751115711715575
  epoch: 5.0


In [ ]:
# ═══════════════════════════════════════════════════════════════════
# CELL 9 — Save model weights immediately after training
# ⚠ Run this as SOON as Cell 8 finishes. Do not delay.
# ═══════════════════════════════════════════════════════════════════

from google.colab import drive

# ─── Mount Google Drive ──────────────────────────────────────────────
# A browser popup will ask you to authorise. Allow it.
drive.mount('/content/drive')

# ─── Save to Google Drive (permanent, survives Colab disconnect) ─────
drive_save_path = '/content/drive/MyDrive/malintent_model'
trainer.save_model(drive_save_path)
tokenizer.save_pretrained(drive_save_path)

print(f"✓ Model saved to Google Drive at: {drive_save_path}")
print("Files:")
!ls -lh {drive_save_path}
# You should see: config.json, model.safetensors (~250MB), tokenizer files

# ─── Also save locally on Colab disk (for download) ─────────────────
# This saves to /content/malintent_model_local/ on the Colab VM
# You'll download this folder to your laptop
local_save_path = './malintent_model_local'
trainer.save_model(local_save_path)
tokenizer.save_pretrained(local_save_path)

print(f"\n✓ Model also saved locally at: {local_save_path}")
print("Files:")
!ls -lh {local_save_path}

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

✓ Model saved to Google Drive at: /content/drive/MyDrive/malintent_model
Files:
total 257M
-rw------- 1 root root  663 May 14 12:29 config.json
-rw------- 1 root root 256M May 14 12:29 model.safetensors
-rw------- 1 root root  328 May 14 12:29 tokenizer_config.json
-rw------- 1 root root 695K May 14 12:29 tokenizer.json
-rw------- 1 root root 5.1K May 14 12:29 training_args.bin


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


✓ Model also saved locally at: ./malintent_model_local
Files:
total 257M
-rw-r--r-- 1 root root  663 May 14 12:29 config.json
-rw-r--r-- 1 root root 256M May 14 12:29 model.safetensors
-rw-r--r-- 1 root root  328 May 14 12:29 tokenizer_config.json
-rw-r--r-- 1 root root 695K May 14 12:29 tokenizer.json
-rw-r--r-- 1 root root 5.1K May 14 12:29 training_args.bin


In [ ]:
# ═══════════════════════════════════════════════════════════════════
# CELL 10 — Final evaluation on the held-out test set
# These are your official metrics for the research paper
# ═══════════════════════════════════════════════════════════════════

import numpy as np
from sklearn.metrics import classification_report, confusion_matrix

# ─── Run predictions on test set ────────────────────────────────────
# trainer.predict() runs inference on the whole test set using the GPU
# Returns: (logits, labels, metrics) — logits have shape (N, 2)
print("Running predictions on test set...")
predictions_output = trainer.predict(test_tokenized)

logits = predictions_output.predictions   # shape: (N, 2) raw scores
labels = predictions_output.label_ids     # shape: (N,)  true labels
preds  = np.argmax(logits, axis=-1)      # shape: (N,)  predicted labels

# ─── Detailed classification report ─────────────────────────────────
# Shows precision, recall, F1 for EACH class (safe and attack)
# Plus weighted averages
print("="*55)
print("           TEST SET EVALUATION RESULTS")
print("="*55)
print(classification_report(labels, preds, target_names=['safe', 'attack'], digits=4))

# ─── Confusion matrix ────────────────────────────────────────────────
# Rows = actual class, Columns = predicted class
# cm[0][0] = True Negatives  (said safe,   actually safe)   ← correct
# cm[0][1] = False Positives (said attack, actually safe)   ← false alarm
# cm[1][0] = False Negatives (said safe,   actually attack) ← DANGEROUS MISS
# cm[1][1] = True Positives  (said attack, actually attack) ← caught!
cm = confusion_matrix(labels, preds)
print("CONFUSION MATRIX")
print("-"*55)
print(f"{'':25s} {'Predicted SAFE':>15} {'Predicted ATTACK':>16}")
print(f"{'Actual SAFE':25s} {cm[0][0]:>15} {cm[0][1]:>16}")
print(f"{'Actual ATTACK':25s} {cm[1][0]:>15} {cm[1][1]:>16}")
print()

# ─── Compute and print key rates ────────────────────────────────────
tn, fp, fn, tp = cm.ravel()
total = tn + fp + fn + tp

print("KEY METRICS SUMMARY")
print("-"*55)
print(f"True Negatives  (safe flagged as safe):    {tn:6d}")
print(f"False Positives (safe flagged as attack):  {fp:6d}  ← false alarms")
print(f"False Negatives (attack missed!):          {fn:6d}  ← ⚠ DANGEROUS")
print(f"True Positives  (attack caught):           {tp:6d}  ← ✓ caught")
print()
print(f"False Positive Rate (false alarm rate): {fp/(fp+tn)*100:.2f}%")
print(f"False Negative Rate (attack miss rate): {fn/(fn+tp)*100:.2f}%  ← keep this LOW")
print()

# ─── Final F1 verdict ────────────────────────────────────────────────
from sklearn.metrics import f1_score
f1 = f1_score(labels, preds, average='binary')
print(f"FINAL F1 SCORE (attack class): {f1:.4f}")

if f1 >= 0.85:
    print(f"✅ Target achieved! F1={f1:.4f} ≥ 0.85")
elif f1 >= 0.80:
    print(f"⚠ F1={f1:.4f}. Close to target. See Day 5 remediation steps below.")
else:
    print(f"❌ F1={f1:.4f} below target. Must investigate — see remediation steps.")

Running predictions on test set...


           TEST SET EVALUATION RESULTS
              precision    recall  f1-score   support

        safe     0.9012    0.8514    0.8756      2099
      attack     0.8591    0.9066    0.8822      2099

    accuracy                         0.8790      4198
   macro avg     0.8802    0.8790    0.8789      4198
weighted avg     0.8802    0.8790    0.8789      4198

CONFUSION MATRIX
-------------------------------------------------------
                           Predicted SAFE Predicted ATTACK
Actual SAFE                          1787              312
Actual ATTACK                         196             1903

KEY METRICS SUMMARY
-------------------------------------------------------
True Negatives  (safe flagged as safe):      1787
False Positives (safe flagged as attack):     312  ← false alarms
False Negatives (attack missed!):             196  ← ⚠ DANGEROUS
True Positives  (attack caught):             1903  ← ✓ caught

False Positive Rate (false alarm rate): 14.86%
False Negative R

## Final Summary

The optimized DistilBERT classifier achieved strong prompt injection detection performance on the held-out HackAPrompt test set.

Final Results:
- Accuracy: ~87.9%
- Attack Recall: ~90.7%
- Attack F1 Score: ~88.2%
- False Negative Rate: ~9.3%

The model prioritises high recall for attack detection, aligning with the security-first design goals of the MalIntent architecture.

This trained model will be integrated into the backend inference wrapper (`ml_classifier.py`) as Layer B of the hybrid detection system.